# 03 — Future field methane calibration: candidate sites and measurement schema

**No field methane values are generated here.** The notebook uses remotely sensed habitat information to suggest *potential* future sampling locations for site reconnaissance. Selected points are not guaranteed accessible, safe, representative or on public land. Never deploy automatically without local authorization and ground review. To preserve the scientific design, keep open water, flooded vegetation, cropland and other land types stratified.

In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
import geopandas as gpd
ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
assert (ROOT/'config/settings.json').exists(),f'Missing files at {ROOT}'
sys.path.insert(0,str(ROOT))
from src.hakaluki_core import project_paths,load_boundary
from src.ee_pipeline import initialize,geometry_from_geodataframe,DW_ID
import ee
P=project_paths(ROOT)
CFG=json.loads((ROOT/'config/settings.json').read_text())
initialize(CFG['earth_engine_cloud_project'])
verified=P['boundary']/'hakaluki_verified.geojson'
candidate=P['boundary']/'hakaluki_analysis_candidate.geojson'
assert verified.exists() or candidate.exists(),'Run Notebook 00 first.'
GDF=load_boundary(verified if verified.exists() else candidate)
ROI=geometry_from_geodataframe(GDF)

## Candidate strata from Dynamic World (2022)

Select candidate sites from the *modal classification*, NOT direct confirmation of flood ecology. Class labels: 0 water, 3 flooded vegetation, 4 crops, 5 shrub & scrub. Requested candidates per class are a **demonstration sampling design**, not statistically justified final sample sizes.

In [ ]:
HABITATS={0:'water',3:'flooded_vegetation',4:'crops',5:'shrub_scrub'}
DW=(ee.ImageCollection(DW_ID).filterDate('2022-01-01','2023-01-01')
    .filterBounds(ROI).select('label'))
print('Number of 2022 Dynamic World tiles:',DW.size().getInfo())
assert DW.size().getInfo()>0,'No Dynamic World imagery; investigate collections.'
classification=DW.mode().rename('label').clip(ROI)
points=classification.stratifiedSample(numPoints=0,classBand='label',
    classValues=list(HABITATS),classPoints=[5,8,8,5],
    region=ROI,scale=30,seed=43,geometries=True)
raw=points.getInfo()['features']
rows=[]
for j,f in enumerate(raw,1):
    c=f.get('geometry',{}).get('coordinates',[None,None])
    label=int(f['properties']['label'])
    rows.append({'candidate_id':f'HAK-{j:03d}','longitude':c[0],
        'latitude':c[1],'habitat_from_dynamic_world':HABITATS[label],
        'dw_class_code':label,'status':'PROVISIONAL_SATELLITE_SITE__NOT_VISITED',
        'flux_value':'','water_depth_cm':'','notes':'Verify access, habitat and sampling feasibility'})
CANDIDATES=pd.DataFrame(rows)
pout=P['calibration']/'candidate_sampling_locations_2022.csv'
CANDIDATES.to_csv(pout,index=False)
print('Saved',len(CANDIDATES),'provisional sites:',pout)
display(CANDIDATES.head())

## Visual review of candidate sites

In [ ]:
import geemap
M=geemap.Map()
M.centerObject(ROI,11)
M.addLayer(classification,{'min':0,'max':8,'palette':['#419BDF','#397D49','#88B053','#7A87C6','#E49635','#DFC35A','#C4281B','#A59B8F','#B39FE1']},'2022 Dynamic World modal class')
M.addLayer(points,{'color':'#c72e29'},'Provisional site suggestions')
M.addLayer(ROI,{'color':'#f79c32'},'Boundary under review')
M

## Field integration contract

When direct chamber or eddy-covariance measurements are eventually available, populate `results/calibration/field_flux_template.csv`. Keep measurement time, flux units, habitat, method, and site identifier. Establish independent quality control and avoid conflating methane concentrations (ppb) with methane flux (mass area−1 time−1). A separate modelling notebook should be written only after measured labels exist. No field observations are simulated.